**Camada GOLD**

A Camada Gold representa a fase final do processo de tratamento de dados, na qual as informações já limpas e padronizadas da Camada Silver são organizadas em um modelo estrela (Star Schema). O objetivo desta etapa é estruturar os dados de forma otimizada para análises, consultas e visualizações, facilitando a exploração do desempenho dos clubes, estatísticas das partidas e eventos do jogo.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ================================================================
# 1) CARREGAR TABELAS SILVER
# ================================================================
silver_full         = spark.table("silver_full")
silver_gols         = spark.table("silver_gols")
silver_cartoes      = spark.table("silver_cartoes")
silver_estatisticas = spark.table("silver_estatisticas")

print("Tabelas Silver carregadas.")


# ================================================================
# 2) DIM_TEMPO
# ================================================================
dim_tempo = (
    silver_full
        .select("data_dt")
        .where("data_dt IS NOT NULL")
        .distinct()
        .withColumn("ano", F.year("data_dt"))
        .withColumn("mes", F.month("data_dt"))
        .withColumn("dia", F.dayofmonth("data_dt"))
        .withColumn("dia_semana", F.date_format("data_dt", "E"))
)

dim_tempo.write.format("delta").mode("overwrite").saveAsTable("dim_tempo")
print("dim_tempo criada.")


# ================================================================
# 3) DIM_CLUBE
# ================================================================
window_clube = Window.orderBy("clube")

dim_clube = (
    silver_full.select(F.col("mandante").alias("clube"))
    .union(silver_full.select(F.col("visitante").alias("clube")))
    .union(silver_gols.select("clube"))
    .union(silver_cartoes.select("clube"))
    .union(silver_estatisticas.select("clube"))
    .where("clube IS NOT NULL")
    .distinct()
    .withColumn("clube_id", F.row_number().over(window_clube))
)

dim_clube.write.format("delta").mode("overwrite").saveAsTable("dim_clube")
print("dim_clube criada.")


# ================================================================
# 4) DIM_ARENA
# ================================================================
window_arena = Window.orderBy("arena")

dim_arena = (
    silver_full
        .select("arena")
        .where("arena IS NOT NULL")
        .distinct()
        .withColumn("arena_id", F.row_number().over(window_arena))
)

dim_arena.write.format("delta").mode("overwrite").saveAsTable("dim_arena")
print("dim_arena criada.")


# ================================================================
# 5) CARREGAR DIMENSÕES
# ================================================================
df_tempo = spark.table("dim_tempo")
df_clube = spark.table("dim_clube")
df_arena = spark.table("dim_arena")


# ================================================================
# 6) FATO PARTIDA
# ================================================================
fato_partida = (
    silver_full.alias("p")
        .join(df_tempo.alias("t"), "data_dt", "left")
        .join(df_clube.alias("c_m"), F.col("p.mandante") == F.col("c_m.clube"), "left")
        .join(df_clube.alias("c_v"), F.col("p.visitante") == F.col("c_v.clube"), "left")
        .join(df_arena.alias("a"), F.col("p.arena") == F.col("a.arena"), "left")
        .select(
            "p.partida_id",
            "p.data_dt",
            "p.data_hora_ts",
            "p.mandante",
            F.col("c_m.clube_id").alias("mandante_id"),
            "p.visitante",
            F.col("c_v.clube_id").alias("visitante_id"),
            "p.arena",
            F.col("a.arena_id"),
            "p.mandante_placar",
            "p.visitante_placar",
            "p.vencedor",
            "p.rodata"
        )
)

fato_partida.write.format("delta").mode("overwrite").saveAsTable("fato_partida")
print("fato_partida criada.")


# ================================================================
# 7) FATO GOL
# ================================================================
fato_gol = (
    silver_gols.alias("g")
        .join(df_clube.alias("c"), "clube", "left")
        .join(silver_full.select("partida_id", "data_dt"), "partida_id", "left")
        .join(df_tempo.alias("t"), "data_dt", "left")
        .select(
            "partida_id",
            "clube",
            "clube_id",
            "minuto",
            "minuto_int",
            "tipo_de_gol",
            "rodata"
        )
)

fato_gol.write.format("delta").mode("overwrite").saveAsTable("fato_gol")
print("fato_gol criada.")


# ================================================================
# 8) FATO CARTAO
# ================================================================
fato_cartao = (
    silver_cartoes.alias("ca")
        .join(df_clube.alias("c"), "clube", "left")
        .join(silver_full.select("partida_id", "data_dt"), "partida_id", "left")
        .join(df_tempo.alias("t"), "data_dt", "left")
        .select(
            "partida_id",
            "clube",
            "clube_id",
            "cartao",
            "atleta",
            "num_camisa",
            "posicao",
            "minuto",
            "minuto_int",
            "rodata"
        )
)

fato_cartao.write.format("delta").mode("overwrite").saveAsTable("fato_cartao")
print("fato_cartao criada.")


# ================================================================
# 9) FATO ESTATISTICA
# ================================================================
fato_estatistica = (
    silver_estatisticas.alias("e")
        .join(df_clube.alias("c"), "clube", "left")
        .join(silver_full.select("partida_id", "data_dt"), "partida_id", "left")
        .join(df_tempo.alias("t"), "data_dt", "left")
        .select(
            "partida_id",
            "clube",
            "clube_id",
            "chutes",
            "chutes_no_alvo",
            "passes",
            "posse_de_bola_num",
            "precisao_passes_num",
            "faltas",
            "cartao_amarelo",
            "cartao_vermelho",
            "impedimentos",
            "escanteios",
            "rodata"
        )
)

fato_estatistica.write.format("delta").mode("overwrite").saveAsTable("fato_estatistica_time_partida")
print("fato_estatistica_time_partida criada.")

print("\n✔ GOLD criado com sucesso!")


Tabelas Silver carregadas.
dim_tempo criada.


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_clube criada.
dim_arena criada.
fato_partida criada.
fato_gol criada.
fato_cartao criada.
fato_estatistica_time_partida criada.

✔ GOLD criado com sucesso!
